# CROW Gemma LoRA — Kaggle T4×2 training run (measured route)

Runnable form of `training/t4/RUNBOOK.md` (staged 2026-11-12, updated after the
3060 dry-run verdict landed in `submission/tools/gemma_dryrun.txt`).

**MEASURED ROUTE (3060 dry-run, 2026-11-12):** the gemma-QAT checkpoint is **untrainable in
transformers-direct** ("QuantizationMethod.GEMMA do not support training" — measured ValueError)
and the fallback lanes die on it. The working lane is **unsloth dequantize-and-train**:
unsloth 2026.10.11 patches Gemma4 ("Fast Gemma4 patching" — measured banner), dequantizes the
QAT base to 16-bit bf16 automatically (~4.9–5.0 GB measured — fits a T4's 16 GB with headroom).
Completion-only masking VERIFIED: unmasked is exactly the answer letter + `<turn|>` end token.
This notebook **hard-stops if the unsloth lane is not the lane that runs**.

Setup (one-time, before running):
1. Settings → Accelerator → **GPU T4 x2** (1× quota burn; L4×4 burns 2× and is for parity evals only).
2. **Add Input → Competition → `gemma-4-developer-agent`**.
3. **Add Input → Model → Google `gemma-4`** (base weights mount read-only under `/kaggle/input` — no download, no license gate).
4. **Upload** (Input panel → Upload, or attach as a Dataset):
   - `corpus_decisions_gemma.jsonl` (518,215 B, 370 rows — **the training corpus, all D rows**)
   - `corpus_b7_swe.jsonl` (1,816,066 B, 695 rows) and `corpus_signals_gemma.jsonl` (976,347 B, 1,239 rows) — staged for other heads; **0 usable D rows for this trainer** (dry-run measured: 0 of 12 b7 rows match the D-row contract)
   - `train_crow_gemma.py` (from `submission/tools/`)

Honesty law: fit wall-time is **NOT MEASURED** (the dry-run completed 0 optimizer steps); VRAM,
masking, and arch numbers below ARE measured (see the dry-run record). Copy every number this
session prints into `eval/smoke-test.md`.

In [ ]:
# Cell 1 — session check (quota clock starts NOW; do not idle)
import datetime, subprocess, sys
T0 = datetime.datetime.now(datetime.timezone.utc)
print('session start (UTC):', T0.isoformat(timespec='seconds'))
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used',
                      '--format=csv'], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), 'NO CUDA — set Accelerator to GPU T4 x2 and re-run from cell 1'
ngpu = torch.cuda.device_count()
print(f'torch {torch.__version__}, cuda ok, devices={ngpu}')
if ngpu != 2:
    print(f'WARNING: expected T4 x2 (device_count==2), got {ngpu} — confirm the accelerator setting')
# Record the Kaggle UI's remaining GPU quota here when known (M0 exit criterion):
REMAINING_GPU_QUOTA = 'NOT MEASURED — read the Kaggle notebook quota widget and replace this string'
print('remaining GPU quota:', REMAINING_GPU_QUOTA)

## Step (b) — install the stack

The dry-run verdict renamed the lanes: **unsloth is not a fast path any more, it is THE lane** —
it dequantizes the QAT base to bf16 and trains it. The TRL/peft and plain-Trainer fallbacks are
**measured-broken for this checkpoint** (transformers 5.15.1 refuses to train GEMMA-quantized
models). `trl`/`peft`/`bitsandbytes` are installed anyway so the script imports cleanly, but if
the run ever falls through to them, cells 12/14 will hard-stop. Versions unpinned except the one
floor the dry-run measured (transformers >= 5.15.1).

In [ ]:
# Cell 2 — pip install
import subprocess, time
_t = time.time()
for cmd in (
    [sys.executable, '-m', 'pip', 'install', '-q', 'unsloth'],
    [sys.executable, '-m', 'pip', 'install', '-q', 'trl', 'peft', 'bitsandbytes', 'accelerate'],
):
    r = subprocess.run(cmd)
    assert r.returncode == 0, f'pip failed: {cmd}'
print(f'pip installs ok in {time.time()-_t:.0f}s (this session only — not an M0 planning number)')

In [ ]:
# Cell 3 — version + QAT-path gate (MEASURED from the 3060 dry-run)
# transformers 5.15.1 is the version the gemma4 arch + GemmaTokenizer were verified on.
import transformers, importlib.util
tfv = tuple(int(x) for x in transformers.__version__.split('.')[:2])
print('transformers:', transformers.__version__)
assert tfv >= (5, 15), (f'transformers {transformers.__version__} < the measured 5.15.1 — '
                        'gemma4 arch support and the masking check were verified on 5.15.1; '
                        'upgrade before running (pip install -U "transformers>=5.15.1")')
assert importlib.util.find_spec('unsloth') is not None, (
    'unsloth MISSING — STOP. The QAT-direct path would be hit, and it is broken by design: '
    'transformers raises ValueError "QuantizationMethod.GEMMA do not support training" '
    '(measured, submission/tools/gemma_dryrun.txt run 5). unsloth dequantize-and-train is '
    'the only working lane for this checkpoint.')
import unsloth  # noqa: F401  (must precede transformers/peft per the trainer's own import order)
print('QAT-path gate PASS: transformers >= 5.15.1 and unsloth present — the dequantize-and-train lane is available.')
print('reminder: unsloth prints "Model already quantized with gemma. Disabling load_in_4bit" '
      'and switches to 16bit LoRA (measured banner) — that IS the intended route; do not pass --no-4bit.')

## Step (c) — corpus staging + verification

Measured on Rog 2026-11-12 (row-by-row scan) and confirmed by the dry-run: the CROW trainer
consumes only `type:"D"` rows. `corpus_decisions_gemma.jsonl` (518,215 B) = **370 rows, all D —
the training corpus**. `corpus_b7_swe.jsonl` (1,816,066 B, 695 message-format rows) and
`corpus_signals_gemma.jsonl` (976,347 B, 1,239 S rows) carry **0 D rows** — the dry-run measured
`[data] 0 usable rows, 12 rejected` on a b7 sample. They ride along for other heads; 370 of the
2,304 uploaded rows train (16.1%).

In [ ]:
# Cell 4 — locate + verify the corpus files (searches /kaggle/input and /kaggle/working)
import json, os, shutil, glob
W = '/kaggle/working'
FILES = ['corpus_b7_swe.jsonl', 'corpus_decisions_gemma.jsonl',
         'corpus_signals_gemma.jsonl']
def find_one(name):
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True)
            + glob.glob(f'{W}/**/{name}', recursive=True))
    return sorted(hits)[0] if hits else None
for name in FILES:
    src = find_one(name)
    assert src, (f'{name} NOT FOUND — upload it via the Input panel '
                 '(or attach it as a Kaggle Dataset) and re-run')
    dst = os.path.join(W, name)
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    n = sum(1 for ln in open(dst, encoding='utf-8', errors='replace') if ln.strip())
    nd = 0
    for ln in open(dst, encoding='utf-8', errors='replace'):
        ln = ln.strip()
        if ln:
            try:
                if str(json.loads(ln).get('type', '')).upper() == 'D':
                    nd += 1
            except json.JSONDecodeError:
                pass
    print(f'{name}: {os.path.getsize(dst):,} B, {n} rows, {nd} D rows')
d = sum(1 for ln in open(os.path.join(W, 'corpus_decisions_gemma.jsonl'),
                         encoding='utf-8', errors='replace') if ln.strip())
assert d == 370, f'corpus_decisions_gemma.jsonl row count changed: {d} != 370 (measured 2026-11-12) — re-read corpus-build-report.md before training'
CORPUS = os.path.join(W, 'corpus_decisions_gemma.jsonl')
print('training corpus =', CORPUS, '(370 usable D rows expected)')

## Step (d.0) — stage the trainer script + the dry-run-recorded B1 patch

The 3060 dry-run found two one-line trainer bugs it was forbidden to fix (record, "TRAINER BUGS
FOUND"): **B1** — `resolve_targets()` does `return list(target_arg)`, which char-splits a comma
string (measured: `Target modules {'g','v',',','a','_','p','j',...} not found`), so the HOUSE
7-module list would crash peft injection; **B2** — `all-linear` resolves QAT wrapper
`QuantizedLinear` subclasses and peft refuses them (measured ValueError). The correct path is
the HOUSE comma list **plus the B1 split fix**. Cell 5 applies B1 to the **staged copy on
Kaggle only** — `submission/tools/train_crow_gemma.py` is never touched from here; the
canonical fix belongs to the trainer work order.

In [ ]:
# Cell 5 — stage train_crow_gemma.py; apply the measured B1 one-liner to the STAGED COPY only
import py_compile, os, glob, shutil
src = find_one('train_crow_gemma.py')
assert src, 'train_crow_gemma.py NOT FOUND — upload it from submission/tools/ and re-run'
SCRIPT = os.path.join(W, 'train_crow_gemma.py')
if os.path.abspath(src) != os.path.abspath(SCRIPT):
    shutil.copy(src, SCRIPT)
code = open(SCRIPT, encoding='utf-8').read()
BUG = 'return list(target_arg)'
FIX = 'return [s.strip() for s in str(target_arg).split(",") if s.strip()]  # B1 fix (dryrun record 2026-11-12)'
if BUG in code:
    assert code.count(BUG) == 1, 'unexpected: multiple B1 sites'
    code = code.replace(BUG, FIX)
    open(SCRIPT, 'w', encoding='utf-8').write(code)
    print('B1 patch APPLIED to the staged copy (comma-string split); submission/tools/ untouched.')
elif FIX in code:
    print('B1 already fixed in the uploaded script — no patch needed.')
else:
    print('WARNING: neither the B1 bug string nor the fix was found — inspect resolve_targets() manually.')
py_compile.compile(SCRIPT, doraise=True)
print('trainer staged + compiles:', SCRIPT)

## Step (a.4) — base weights at /kaggle/input

Mount shape per the official getting-started notebook cell 4:
`/kaggle/input/models/google/gemma-4/other/{declared_model}/2`. The real path is **discovered**,
not hardcoded. Single-base rule (HARNESS_README §3.2): `agent.yaml` declares
`gemma-4-e2b-it-qat-w4a16-ct` — the checkpoint found here must be the same base. The checkpoint
stays QAT on disk; unsloth dequantizes to bf16 at load (measured banner). VRAM expectation
(measured on the dry-run stand-in at the same E2B scale): 16-bit load ≈ 4.88–5.02 GB torch /
5,349 MiB nvidia-smi peak → fits a 16 GB T4 with headroom.

In [ ]:
# Cell 6 — discover the mounted base checkpoint and assert the single-base match
cands = set()
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    dd = os.path.dirname(cfg)
    low = dd.lower()
    if 'gemma' in low and (glob.glob(dd + '/*.safetensors') or glob.glob(dd + '/**/*.safetensors', recursive=True)):
        cands.add(dd)
assert cands, ('no gemma checkpoint found under /kaggle/input — attach the gemma-4 model input '
               '(Add Input → Models) and re-run from cell 6')
print('checkpoint dirs found:')
for c in sorted(cands):
    print('  ', c)
e2b = [c for c in cands if 'e2b' in c.lower()]
BASE_DIR = sorted(e2b)[0] if e2b else sorted(cands)[0]
assert 'e2b' in BASE_DIR.lower(), (
    f'BASE_DIR={BASE_DIR} is NOT the declared base gemma-4-e2b-it-qat-w4a16-ct '
    '(submission/agent.yaml line 6). STOP: an adapter trained on the wrong base breaks '
    'the single-base rule — attach the e2b variant or re-read the plan before overriding.')
print('BASE_DIR =', BASE_DIR)

In [ ]:
# Cell 7 — in-cloud dry run: base load (unsloth dequantize) + completion-only masking
import subprocess, time
_t = time.time()
r = subprocess.run([sys.executable, SCRIPT, '--dry-run',
                    '--base', BASE_DIR, '--corpus', CORPUS],
                   capture_output=True, text=True)
print(r.stdout[-5000:])
print(r.stderr[-2000:])
assert r.returncode == 0, f'dry-run exited {r.returncode} — fix before the fit (exit 2 = base load failed)'
assert 'unsloth lane ok' in r.stdout, (
    'the run fell through to the transformers/TRL fallback lane. HARD STOP — measured on the '
    '3060 dry-run: that lane cannot train the QAT checkpoint ("QuantizationMethod.GEMMA do '
    'not support training"). Diagnose the unsloth load failure above instead of proceeding.')
assert 'DRY-RUN PASS' in r.stdout, 'dry-run ran but did not print DRY-RUN PASS — read the masking block above'
print(f'in-cloud dry-run ok in {time.time()-_t:.0f}s — record this number in eval/smoke-test.md')

## Step (d) — the fit (HOUSE recipe, unsloth lane)

Flags are the script's own argparse defaults (TEV1_METHOD.md house recipe): r16 alpha16 dropout0
on the 7 attention+MLP projections, lr 2e-4, warmup 10, **2 epochs**, micro 1 × accum 8
(effective 8), seed 3407, max_len 512. 4-bit is REQUESTED but unsloth overrides it to 16-bit
for the QAT base (measured banner: "Model already quantized with gemma. Disabling
`load_in_4bit`… Switching to 16bit LoRA") — do NOT pass `--no-4bit` and do NOT pass
`--packing`. No `--fallback-transformers` (measured-broken on QAT). `save_strategy="no"` — only
the final PEFT save is written. LoRA prep is measured at 37,920,768 trainables / 2,365,851,416
total = 1.603% on the E2B-scale stand-in.

Wall time: **NOT MEASURED** — the dry-run completed 0 optimizer steps. Arithmetic: 370 D rows ×
2 epochs = 740 micro steps ≈ 94 optimizer steps; loss logs print at steps 5/10 per epoch
(`logging_steps=5` is hardcoded). The cell prints the real wall time; paste it into
`eval/smoke-test.md`.

In [ ]:
# Cell 8 — THE FIT (quota burns every second this cell runs)
import subprocess, time, json
OUT = os.path.join(W, 'adapters', 'crow_lora')
STATE = os.path.join(W, 'train_state_crow_gemma.json')
LOG = os.path.join(W, 'fit_log.txt')
cmd = [sys.executable, SCRIPT,
       '--base', BASE_DIR, '--corpus', CORPUS,
       '--out', OUT, '--state', STATE,
       '--r', '16', '--alpha', '16', '--dropout', '0.0',
       '--target-modules', 'q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj',
       '--lr', '2e-4', '--warmup', '10', '--epochs', '2',
       '--micro', '1', '--accum', '8', '--seed', '3407', '--max-len', '512']
print(' '.join(cmd))
_t = time.time()
with open(LOG, 'w', encoding='utf-8') as fh:
    r = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
FIT_WALL_S = time.time() - _t
logtxt = open(LOG, encoding='utf-8', errors='replace').read()
print(logtxt[-5000:])
print(f'FIT WALL TIME (measured, this session): {FIT_WALL_S:.0f}s = {FIT_WALL_S/60:.1f} min')
assert r.returncode == 0, f'fit exited {r.returncode} (2 = base load failed, 3 = fit failed) — see fit_log.txt'
assert 'unsloth lane ok' in logtxt, (
    'the fit ran on the fallback lane. HARD STOP — measured: the fallback lanes cannot train '
    'the QAT checkpoint. If an adapter was still written, treat it as suspect and re-run on '
    'the unsloth lane.')
st = json.load(open(STATE))
assert st.get('status') == 'done', f'train_state says {st!r}, expected status=done'
print('train_state:', st)

In [ ]:
# Cell 9 — post-run verification (RUNBOOK step f)
import os, json
cfg = json.load(open(os.path.join(OUT, 'adapter_config.json')))
assert cfg.get('r') == 16 and cfg.get('lora_alpha') == 16, f'rank mismatch: r={cfg.get("r")} alpha={cfg.get("lora_alpha")}'
assert float(cfg.get('lora_dropout', -1)) == 0.0, f'dropout={cfg.get("lora_dropout")}'
tm = cfg.get('target_modules')
assert set(tm) == {'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'}, f'target_modules={tm}'
sft = [f for f in os.listdir(OUT) if f.endswith('.safetensors')]
assert 'adapter_config.json' in os.listdir(OUT) and sft, 'PEFT layout incomplete (HARNESS_README 3.4)'
total = sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT) if os.path.isfile(os.path.join(OUT, f)))
assert total < 3 * 1024**3, f'adapter dir {total:,} B exceeds the 3 GiB submission budget'
print('adapter_config: r=16 alpha=16 dropout=0.0 targets=7 projections  PASS')
print(f'adapter dir total: {total:,} B ({total/2**20:.1f} MiB) — HARNESS_README 3.4 rank-16 envelope is ~110-220 MB')
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f'  {f}  {os.path.getsize(p):,} B')

## Step (f) — checkpoint download

Zip the adapter dir into `/kaggle/working`, print SHA-256 + size, then download via the notebook
**Output** panel. On Rog it goes to `submission/adapters/crow_lora/` (PEFT layout, discovered by
the harness adapter scan) and must pass the **C0 gate** (`submission/eval-split.md`: adapter arm
STRICTLY > baseline arm, both arms twice) before anything ships. Do not download the base
weights.

In [ ]:
# Cell 10 — zip + digest for download
import zipfile, hashlib
Z = os.path.join(W, 'crow_lora.zip')
with zipfile.ZipFile(Z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT):
        for f in files:
            p = os.path.join(root, f)
            zf.write(p, os.path.relpath(p, W))
h = hashlib.sha256(open(Z, 'rb').read()).hexdigest()
print('download:', Z)
print('size:', f'{os.path.getsize(Z):,} B')
print('sha256:', h)
print('FIT_WALL_S (paste into eval/smoke-test.md):', round(FIT_WALL_S, 1), 's')

## Step (g) — kill/cleanup discipline

1. **Save Version** (Save & Run All) so the log + zip are captured.
2. **Then explicitly stop the session** (⋯ → Stop). Closing the tab does not free the GPU; every idle attached minute burns 1× of the weekly pool.
3. Keep `/kaggle/working` lean: adapter zip, `fit_log.txt`, `train_state_crow_gemma.json` only.
4. Record into `eval/smoke-test.md` right after: fit wall time, dry-run time, pip-install time, `train_state` final loss/step, the adapter byte size, and the Kaggle UI's remaining GPU quota. If observed burn ≠ 1× for T4×2, edit GEMMA_PLAN.md's COMPUTE table to the measured values (KAGGLE_EXPECTATIONS fix 2).

In [ ]:
# Cell 11 — final report + stop reminder
import torch, datetime
free, total_mem = torch.cuda.mem_get_info(0)
peak = torch.cuda.max_memory_allocated(0) / 2**30
print(f"gpu0 peak={peak:.2f}GB free={free/2**30:.2f}/{total_mem/2**30:.2f}GB")
print('session end (UTC):', datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'))
print('NOW: Save Version, then STOP the session (⋯ → Stop). The quota clock is still running.')